In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum as _sum, unix_timestamp, collect_set, size

def test_fraud_detection(spark):
    data = [
        ("C1", "TX1", 20000, "Mumbai", "2024-01-01 10:00:00"),
        ("C1", "TX2", 20000, "Delhi", "2024-01-01 10:05:00"),
        ("C1", "TX3", 15000, "Mumbai", "2024-01-01 10:08:00"),
        ("C2", "TX1", 20000, "Mumbai", "2024-01-01 10:08:00")
    ]

    columns = ["customer_id", "txn_id", "amount", "location", "txn_time"]
    df = spark.createDataFrame(data, columns)

    df = df.withColumn("txn_ts", unix_timestamp("txn_time"))

    window_spec = Window.partitionBy("customer_id") \
        .orderBy("txn_ts") \
        .rangeBetween(-600, 0)

    result_df = df.withColumn("total_amount", _sum("amount").over(window_spec)) \
                  .withColumn("locations_set", collect_set("location").over(window_spec)) \
                  .withColumn("distinct_locations", size(col("locations_set"))) \
                  .filter((col("total_amount") > 50000) & (col("distinct_locations") > 1))

    result = {(r["customer_id"], r["txn_id"]) for r in result_df.collect()}

    expected = {("C1", "TX3")}

    try:
        assert result == expected
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")
        print("Expected:", expected)
        print("Got:", result)

test_fraud_detection(spark)